In [ ]:
# Load Dataset
from pathlib import Path
import pandas as pd
from tkinter import Tk, filedialog

# Chọn trực tiếp file breathing_frequency.xlsx (4 file phải nằm cùng thư mục)
root = Tk()
root.withdraw()
selected_file = filedialog.askopenfilename(
    title="Chọn file breathing_frequency.xlsx",
    filetypes=[("Excel files", "*.xlsx"), ("All files", "*.*")]
)
root.destroy()

if not selected_file:
    raise FileNotFoundError("Chưa chọn file breathing_frequency.xlsx")

RAW_DIR = Path(selected_file).parent

breathing_path = RAW_DIR / "breathing_frequency.xlsx"
hrv_path = RAW_DIR / "hrv_indexes.xlsx"
rsa_path = RAW_DIR / "raw_data_RSA.xlsx"
rr_path = RAW_DIR / "RR_long_format.csv"

required_files = [breathing_path, hrv_path, rsa_path, rr_path]
missing_files = [p.name for p in required_files if not p.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Thư mục đã chọn không có đủ 4 file. Thiếu: "
        + ", ".join(missing_files)
        + f"\\nThư mục đang chọn: {RAW_DIR}"
    )

# Load source files
breathing_df = pd.read_excel(breathing_path)
hrv_df = pd.read_excel(hrv_path)
rr_df = pd.read_csv(rr_path)

rsa_excel = pd.ExcelFile(rsa_path)
rsa_raw = pd.read_excel(
    rsa_path,
    sheet_name=rsa_excel.sheet_names[0],
    header=None
)

# Chuẩn hóa tên cột RR để các cell phân tích dùng thống nhất
def normalize_column_name(df, target, candidates):
    if target in df.columns:
        return
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            df.rename(columns={lookup[candidate.lower()]: target}, inplace=True)
            return

normalize_column_name(rr_df, "rr", [
    "rr_ms", "rr interval", "rr_interval", "rr_intervals",
    "rr interval (ms)", "RR"
])
normalize_column_name(rr_df, "timestamps", [
    "timestamp_s", "timestamp", "time_s", "time", "timestamps_s"
])
normalize_column_name(rr_df, "protocol_step", [
    "protocol_step_bpm", "protocol", "step"
])

# Chuẩn hóa kiểu số nếu có cột RR
if "rr" in rr_df.columns:
    rr_df["rr"] = pd.to_numeric(rr_df["rr"], errors="coerce")

print("RAW DATA FOLDER:", RAW_DIR)
print("Breathing sheets:", pd.ExcelFile(breathing_path).sheet_names)
print("HRV sheets:", pd.ExcelFile(hrv_path).sheet_names)
print("RSA sheets:", rsa_excel.sheet_names)
print("\\nDATASETS LOADED")
print("=" * 60)
print("Breathing:", breathing_df.shape)
print("HRV:", hrv_df.shape)
print("RR intervals:", rr_df.shape)
print("RSA raw:", rsa_raw.shape)
print("\\nRR columns:", rr_df.columns.tolist())


In [1]:
# Dataset đã được load ở Cell 1 phía trên.
# Không chạy lại cell này; cell này được giữ làm ghi chú để tránh nạp dữ liệu bằng đường dẫn cũ.


FileNotFoundError: Không tìm thấy breathing_frequency.xlsx. Kiểm tra lại thư mục giải nén data.

In [ ]:
# Dataset Overview
datasets = {
    "Breathing Frequency": breathing_df,
    "HRV Indexes": hrv_df,
    "RR Intervals": rr_df,
    "RSA Raw": rsa_raw
}

overview = []

for name, df in datasets.items():
    overview.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Memory (MB)": round(
            df.memory_usage(deep=True).sum() / 1024**2, 3
        )
    })

overview_df = pd.DataFrame(overview)

display(overview_df)

In [ ]:
#  Inspect Data Structure
print("BREATHING FREQUENCY")
display(breathing_df.head())

print("\nHRV INDEXES")
display(hrv_df.head())

print("\nRR INTERVALS")
display(rr_df.head())

print("\nRSA RAW DATA")
display(rsa_raw.iloc[:8, :12])

In [ ]:
# Column Information
for name, df in datasets.items():

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    print("Shape:", df.shape)

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum().sort_values(ascending=False).head(15))

In [ ]:
#  Missing Values Analysis
missing_results = []

for name, df in datasets.items():

    total_cells = df.size
    missing = int(df.isna().sum().sum())

    missing_results.append({
        "Dataset": name,
        "Total Cells": total_cells,
        "Missing Cells": missing,
        "Missing Percentage": round(
            missing / total_cells * 100, 3
        ) if total_cells else 0
    })

missing_df = pd.DataFrame(missing_results)

display(missing_df)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=missing_df,
    x="Dataset",
    y="Missing Percentage"
)

plt.title("Missing Values by Dataset")
plt.ylabel("Missing Values (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show() 

NameError: name 'datasets' is not defined

In [ ]:
# Statistical Summary
print("BREATHING FREQUENCY")
display(breathing_df.describe().T)

print("\nHRV STATISTICS")
display(hrv_df.describe().T)

print("\nRR INTERVAL STATISTICS")
display(rr_df.describe().T)

In [ ]:
 # Breathing Frequency Analysis
breathing_columns = [
    col for col in breathing_df.columns
    if col.startswith("bb_")
]

print("Breathing frequency columns:")
print(breathing_columns)

breathing_summary = breathing_df[breathing_columns].describe().T

display(breathing_summary)

plt.figure(figsize=(12, 6))

breathing_long = breathing_df[
    breathing_columns
].melt(
    var_name="Protocol",
    value_name="Breathing Frequency"
)

sns.boxplot(
    data=breathing_long,
    x="Protocol",
    y="Breathing Frequency"
)

plt.title("Breathing Frequency Distribution")
plt.xlabel("Protocol")
plt.ylabel("Breathing Frequency (dataset units)")
plt.tight_layout()
plt.show()

In [ ]:
# HRV Analysis
hrv_features = [
    "MeanHR",
    "mRR",
    "SDNN",
    "RMSSD",
    "pNN50",
    "LF_HF_ratio",
    "Total_power"
]

available_hrv = [
    col for col in hrv_features
    if col in hrv_df.columns
]

display(hrv_df[available_hrv].describe().T)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.histplot(
    hrv_df["MeanHR"].dropna(),
    kde=True,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Mean Heart Rate")

sns.histplot(
    hrv_df["SDNN"].dropna(),
    kde=True,
    ax=axes[0, 1]
)
axes[0, 1].set_title("SDNN Distribution")

sns.histplot(
    hrv_df["RMSSD"].dropna(),
    kde=True,
    ax=axes[1, 0]
)
axes[1, 0].set_title("RMSSD Distribution")

sns.histplot(
    hrv_df["LF_HF_ratio"].dropna(),
    kde=True,
    ax=axes[1, 1]
)
axes[1, 1].set_title("LF/HF Ratio")

plt.tight_layout()
plt.show()

In [ ]:
 #  RR Interval Analysis
rr_df["rr"] = pd.to_numeric(
    rr_df["rr"],
    errors="coerce"
)

rr_df["HR_bpm"] = 60000 / rr_df["rr"]

print("RR interval statistics:")
display(rr_df["rr"].describe())

print("\nCalculated heart rate:")
display(rr_df["HR_bpm"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    rr_df["rr"].dropna(),
    bins=50,
    kde=True,
    ax=axes[0]
)

axes[0].set_title("RR Interval Distribution")
axes[0].set_xlabel("RR Interval (ms)")

sns.histplot(
    rr_df["HR_bpm"].dropna(),
    bins=50,
    kde=True,
    ax=axes[1]
)

axes[1].set_title("Calculated Heart Rate")
axes[1].set_xlabel("Heart Rate (bpm)")

plt.tight_layout()
plt.show()
 

In [ ]:
# RR Signal Visualization
# Select one participant and session
participant = rr_df["participant_id"].iloc[0]
session = rr_df["session_id"].iloc[0]

sample_signal = rr_df[
    (rr_df["participant_id"] == participant) &
    (rr_df["session_id"] == session)
].copy()

sample_signal = sample_signal.sort_values("timestamps")

plt.figure(figsize=(15, 5))

plt.plot(
    sample_signal["timestamps"],
    sample_signal["rr"],
    linewidth=0.8
)

plt.title(
    f"RR Interval Sequence - Participant {participant}, {session}"
)

plt.xlabel("Timestamp (seconds)")
plt.ylabel("RR Interval (ms)")

plt.tight_layout()
plt.show() 

In [ ]:
# Correlation Analysis
correlation_features = [
    "MeanHR",
    "mRR",
    "SDNN",
    "RMSSD",
    "pNN50",
    "LF_HF_ratio",
    "Total_power"
]

correlation_features = [
    col for col in correlation_features
    if col in hrv_df.columns
]

corr = hrv_df[correlation_features].corr(
    method="spearman"
)

plt.figure(figsize=(11, 8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("HRV Feature Correlation - Spearman")
plt.tight_layout()
plt.show()

In [ ]:
# Protocol Analysis
if "protocol_step" in hrv_df.columns:
    print("HRV protocol distribution")
    display(
        hrv_df.groupby("protocol_step").agg(
            participants=("participant_id", "nunique"),
            records=("participant_id", "size"),
            MeanHR=("MeanHR", "mean"),
            SDNN=("SDNN", "mean"),
            RMSSD=("RMSSD", "mean")
        )
    )
else:
    print("HRV: không có cột protocol_step; bỏ qua phân tích theo protocol.")

if "protocol_step" in rr_df.columns:
    print("\\nRR protocol distribution")
    rr_agg = {
        "participants": ("participant_id", "nunique"),
        "records": ("rr", "size"),
        "mean_RR": ("rr", "mean")
    }
    if "HR_bpm" in rr_df.columns:
        rr_agg["mean_HR"] = ("HR_bpm", "mean")
    display(rr_df.groupby("protocol_step").agg(**rr_agg))
else:
    print("\\nRR: không có cột protocol_step; bỏ qua phân tích theo protocol.")


In [ ]:
# Participant Analysis
participant_summary = rr_df.groupby(
    "participant_id"
).agg(
    RR_Count=("rr", "count"),
    Mean_RR=("rr", "mean"),
    Median_RR=("rr", "median"),
    Mean_HR=("HR_bpm", "mean"),
    Sessions=("session_id", "nunique")
).reset_index()

display(participant_summary.head(10))

plt.figure(figsize=(14, 5))

sns.boxplot(
    data=rr_df,
    x="participant_id",
    y="HR_bpm"
)

plt.title("Heart Rate Distribution by Participant")
plt.xlabel("Participant ID")
plt.ylabel("Heart Rate (bpm)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Data Quality Checks
quality_report = []

# RR interval
quality_report.append({
    "Check": "RR values positive",
    "Result": bool((rr_df["rr"].dropna() > 0).all())
})

quality_report.append({
    "Check": "RR values finite",
    "Result": bool(np.isfinite(rr_df["rr"].dropna()).all())
})

# HRV
quality_report.append({
    "Check": "HRV numerical values finite",
    "Result": bool(
        np.isfinite(
            hrv_df.select_dtypes(include=np.number)
        ).all().all()
    )
})

# Participant consistency
quality_report.append({
    "Check": "RR participant IDs present",
    "Result": bool(rr_df["participant_id"].notna().all())
})

quality_report.append({
    "Check": "HRV participant IDs present",
    "Result": bool(hrv_df["participant_id"].notna().all())
})

display(pd.DataFrame(quality_report))

In [ ]:
# Save Inspection Results
overview_df.to_csv(
    PROCESSED_DIR / "dataset_overview.csv",
    index=False
)

missing_df.to_csv(
    PROCESSED_DIR / "missing_values_report.csv",
    index=False
)

breathing_summary.to_csv(
    PROCESSED_DIR / "breathing_frequency_statistics.csv"
)

participant_summary.to_csv(
    PROCESSED_DIR / "participant_summary.csv",
    index=False
)

corr.to_csv(
    PROCESSED_DIR / "hrv_correlation.csv"
)

pd.DataFrame(quality_report).to_csv(
    PROCESSED_DIR / "data_quality_report.csv",
    index=False
)

print("Inspection reports saved successfully!")

In [ ]:
# Key Findings & Conclusion
print("=" * 70)
print("RESPIRATORY DATA INSPECTION - FINAL REPORT")
print("=" * 70)

print("1. DATASET INFORMATION")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

print("\n2. PARTICIPANTS")

print(
    "RR participants:",
    rr_df["participant_id"].nunique()
)

print(
    "HRV participants:",
    hrv_df["participant_id"].nunique()
)

print(
    "Breathing participants:",
    breathing_df["participant_id"].nunique()
)

print("\n3. SIGNAL INFORMATION")

print("RR intervals: available")
print("Heart rate derived from RR: available")
print("HRV features: available")
print("Breathing frequency features: available")
print("Continuous respiratory waveform: not available")

print("\n4. DATA QUALITY")

print("RR missing:", rr_df["rr"].isna().sum())
print("HRV missing:", hrv_df.isna().sum().sum())
print(
    "Breathing missing:",
    breathing_df.isna().sum().sum()
)

print("\n5. CONCLUSION")

print(
    "Dataset is suitable for exploratory physiological analysis."
)

print(
    "Continuous respiratory waveform is required "
    "before training a waveform-based CNN."
)

print(
    "No disease diagnosis labels have been established."
)